In [1]:
import numpy as np

In [2]:
# States: (Load, Position)
states = ["Idle_A", "Loaded_A", "Idle_B", "Loaded_B"]

# Actions: Robot decisions
actions = ["Pick", "Move", "Charge"]

n_states = len(states)
n_actions = len(actions)

In [3]:
# P[action][state][next_state]
P = np.array([
    # Pick
    [[0.1, 0.9, 0.0, 0.0],
     [0.0, 1.0, 0.0, 0.0],
     [0.0, 0.0, 0.1, 0.9],
     [0.0, 0.0, 0.0, 1.0]],

    # Move
    [[0.0, 0.0, 1.0, 0.0],
     [0.0, 0.0, 0.0, 1.0],
     [1.0, 0.0, 0.0, 0.0],
     [0.0, 1.0, 0.0, 0.0]],

    # Charge
    [[1.0, 0.0, 0.0, 0.0],
     [0.5, 0.5, 0.0, 0.0],
     [0.0, 0.0, 1.0, 0.0],
     [0.0, 0.0, 0.5, 0.5]]
])

In [4]:
# Reward: higher = faster task completion
R = np.array([
    [5, -1, 5, -1],   # Pick
    [2, 2, 2, 2],     # Move
    [-2, 3, -2, 3]    # Charge
])

In [5]:
policy = np.zeros(n_states, dtype=int)  # start with action 0 (Pick)
V = np.zeros(n_states)

gamma = 0.9
theta = 0.001

In [6]:
def policy_evaluation(policy, V):
    while True:
        delta = 0
        new_V = np.copy(V)

        for s in range(n_states):
            a = policy[s]
            value = 0

            for s_next in range(n_states):
                value += P[a][s][s_next] * (R[a][s] + gamma * V[s_next])

            new_V[s] = value
            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V

        if delta < theta:
            break

    return V

In [7]:
def policy_improvement(V, policy):
    policy_stable = True

    for s in range(n_states):
        old_action = policy[s]
        action_values = []

        for a in range(n_actions):
            value = 0
            for s_next in range(n_states):
                value += P[a][s][s_next] * (R[a][s] + gamma * V[s_next])
            action_values.append(value)

        best_action = np.argmax(action_values)
        policy[s] = best_action

        if old_action != best_action:
            policy_stable = False

    return policy, policy_stable

In [8]:
while True:
    V = policy_evaluation(policy, V)
    policy, stable = policy_improvement(V, policy)

    if stable:
        break

In [9]:
action_names = ["Pick", "Move", "Charge"]

print("Optimal Value Function:")
for i in range(n_states):
    print(f"{states[i]}: {V[i]:.2f}")

print("\nOptimal Policy:")
for i in range(n_states):
    print(f"{states[i]} → {action_names[policy[i]]}")

Optimal Value Function:
Idle_A: 38.08
Loaded_A: 36.61
Idle_B: 38.08
Loaded_B: 36.61

Optimal Policy:
Idle_A → Pick
Loaded_A → Charge
Idle_B → Pick
Loaded_B → Charge
